회귀모델은 추세(Trend)와 계절성을 사람이 변수로 만들어서 입력해야 한다.

시계열 전용 모델이 있는데, 시계열 전용 모델은 최근 값과 최근 오차가 다음값에 어떻게 이어지는가를 데이터에서 스스로 찾는다(학습한다).

- SARIMA
- 지수평활(Exponential Smoothing)


## SARIMA

ARIMA, AR(자기회귀, AutoRegressive), I(누적, Integrated), MA(이동평균 오차, Moving Average)

자기회귀는 지난달 값이 높았으면 이번 달도 높다처럼 과거 값으로 현재를 설명하는것이고, 누적은 값 대신 전월과의 차이(차분)를 모델링해서 추세를 없애고,  이동평균은 지난달 예측이 빗나간만큼 이번달 값을 보정하겠다.



# 모델의 평가 설계, 개선

모델이 예측한 결과를 얼마나 믿을 수 있는지를 따져볼 수 있는 도구가 필요하다. 이때, RMSE, ME, MASE같은 평가 지표를 더하고 예측기간별로 오차를 나누어보는 롤링 원점 평가를 설계하고, 로그 변화 회귀, 계절 시차 특성, 학습 기간 선택, 예측/결합 등의 방법으로 모델을 다듬을 수 있다.


## 지표

- RMSE(Root Mean Squared Error, 평균 제곱근 오차): 큰 오차가 얼마나 있는가?
- ME(Mean Error, 편향): 한쪽으로 치우쳐져서 예측하는가?
- MASE(Mean Absolute Scaled Error): 계절 나이브보다얼마나 나은가

### RMSE

RMSE는 오차를 제곱해서 평균을 구한 다음에 다시 제곱근을 씌운 값을 의미한다. 제곱을 하게되면 큰 오차가 더 커지게 된다. 그래서 RMSE를 구하게되면 큰 오차에 민감하게 반응할 수 있게 된다.



In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression

plt.rcParams['font.family'] = 'Malgun Gothic'


def load_dataset(filename):
    BASE_DIR = './data'
    EXTENSION = '.csv'
    TARGET_PATH = f'{BASE_DIR}/{filename}{EXTENSION}'
    try:
        return pd.read_csv(TARGET_PATH)
    except FileNotFoundError:
        return None
    except UnicodeDecodeError:
        return pd.read_csv(TARGET_PATH, encoding='euc-kr')


In [4]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# RMSE, MAE: 늘 조금씩 틀리는 예측, 가끔 크게 틀리는 예측

actual = np.full(12, 2_000_000)

steady = actual + np.array([50_000, -50_000] * 6)

spiky = actual.copy()
spiky[6] = actual[6] + 550_000
spiky = spiky + np.array([1_000] * 6 + [0] + [1_000] * 5)

errors = actual - spiky
print(f'MAE = {round(np.abs(errors).mean())}')
# RMSE
# sqrt()
print(f'RMSE = {round(np.sqrt((errors ** 2).mean()))}')

MAE = 46750
RMSE = 158774


In [7]:
for name, pred in [ ('늘 조금씩 틀림', steady), ('가끔 크게 틀림', spiky) ]:
    mae = mean_absolute_error(actual, pred)
    rmse = root_mean_squared_error(actual, pred)
    print(f'{name} 모델의  MAE = {mae:,.0f} RMSE = {rmse:,.0f}')

늘 조금씩 틀림 모델의  MAE = 50,000 RMSE = 50,000
가끔 크게 틀림 모델의  MAE = 46,750 RMSE = 158,774


## 편향(Bias)

예측에서도 오차가 위아래로 고르게  흩어지는 것과, 한쪽으로 몰릴때 조치해야되는 내용이 다르다. 한쪽으로 치우친 정도를 편향(Bias)이라고 일컫는다. 오차에 절댓값을 씌우지 않고 그대로 평균을 구하게되면 평균 오차(ME, Mean Error)라고 일컫는다.

편향이 양수이면 모델이 실제보다 작게 예측하는 경향이 있고, 음수면 크게 예측하는 경향이 있다. MAE 가운데 평향이 차지하는 몫이 크면 모델이 추세나 수준을 잘못잡고있다 라고 판단할수있고, 이러면 모델 구조를 고쳐야된다. 편향이 0에 가깝고 MAE만 크면 설명하지 못하는 불규칙 변동이 크다고 볼 수 있다.

10만, -10만의 오차가 합산되면 0이된다.  편향이 없는것 처럼 보인다. 모델의 예측은 정확하지 않는 것이다. 편향은 편향만 구해서 사용하는것이 아니라 MAE와 같이 사용한다.

- 편향 = 평균(실제값 - 예측값)
  - 양수: 과소 예측
  - 음수: 과대 예측

## MASE(Mean Absolute Scaled Error)

MAE가 5만 TEU라는 결과가 나왔을때, 해당 지표가 좋은 지표인지 나쁜 지표인지를 판단하기 어렵다. 월 물동량 200만 TEU에서는 오차가 2.5%정도이기 때문에 좋은 상태라고 판단해볼 수 있지만, 다른 수단으로 4만 TEU를 쉽게 구할수 있었으면 이 지표는 좋지 않은 지표로 볼 수 있는 것이다.

MASE는 이러한 비교를 한 숫자로 만든 지표로서 모델의 MAE를 계절 나이브 학습 구간에서 보인 평균 오차로 나눈 값.


In [13]:
season = [ -40, -120, 60, 40, 50, 10, 30, -10, -60, 10, -10, -10 ]

noise = np.random.default_rng(42).normal(0, 15, 48).round(0)
idx = pd.date_range('2021-01-01', periods=48, freq='MS')

toy = pd.Series([ 1900 + 5 * i + season[i % 12] for i in range(48)], index=idx)

In [17]:
train, test = toy.iloc[:36], toy.iloc[36:]

In [18]:
# 분모: 학습 구간에서 계절 나이브(12개월전 값)의 평균 절대 오차
scale = (train - train.shift(12)).abs().mean()
print(round(scale, 1))

60.0


In [19]:
def mase(actual, pred, scale):
    return mean_absolute_error(actual, pred) / scale

In [20]:
seasonal_naive = train.iloc[-12:].to_numpy() # 작년 같은달 값
with_growth = seasonal_naive + 12 * 5 # 작년 같은 달 + 1년치 추세(월 5천 TEU * 12)

In [22]:
mase(test, seasonal_naive, scale)
mase(test, with_growth, scale)

np.float64(0.0)

## 예측기간(Horizon)

예측기간은 예측을 만든 시점에서 몇 칸앞의 값을 예측하는가를 의미한다.

2월 말에 3월을 예측하면 1개월 앞에 있는 예측(h=1), 다음해 2월의 값을 예측 12개월(h=12)이 된다.

일기예보를 두고볼때 내일것은 그래도 잘 맞는 편임 열흘뒤의 예보를 보면 상대적으로 잘 안맞는다. 예측은 먼 미래일수록 그 사이에 생길 수있는 일이 많아지기 때문에 오차가 커질수밖에 없다.

예측 오차는 예측 기간별로 따로따로 살펴보아야한다. MAE가 5만이다. 보다는 1개월 앞의 MAE가 4만, 12개월 앞의 MAE는 7만이라는 내용이 잘 맞는 표현이다.



In [24]:
origin = train.index[-1] # 예측을 만드는 시점
print(origin)

2023-12-01 00:00:00


In [26]:
horizon = np.arange(1, 13)
print(horizon)

[ 1  2  3  4  5  6  7  8  9 10 11 12]


In [28]:
train.iloc[-1]

np.int64(2065)

In [27]:
naive = np.full(12, train.iloc[-1])
print(naive)

[2065 2065 2065 2065 2065 2065 2065 2065 2065 2065 2065 2065]


In [29]:
abs_err = pd.DataFrame({
    'h': horizon,
    'naive': np.abs(test.to_numpy() - naive),
    'seasonality naive': np.abs(test.to_numpy() - seasonal_naive)
}).set_index('h').round(0)

abs_err

,naive,seasonality naive
h,,
1,25,60
2,100,60
3,85,60
4,70,60
5,85,60
6,50,60
7,75,60
8,40,60
9,5,60


## 롤링 원점 평가

롤링원점평가는 예측을 시작하는 시점(원점)을 한달씩 뒤로 옮기면서, 매번 원점까지의 데이터 학습, 원점 뒤 h개월 예측, 실제값과 비교를 반복.


In [31]:
def seasonal_naive_forecast(history, horizon):
    return history.iloc[-12:].to_numpy()[:horizon]

def rolling_origin(s, forecaster, origins, horizon = 12):
    records = []

    for origin in origins:
        history = s.loc[:origin]
        future = s.loc[origin:]
        pred = forecaster(history, horizon)[:len(future)]
        for h, (date, actual_value) in enumerate(future.items(), start=1):
            records.append({'원점': origin, 'h': h, '날짜': date, '실제값': actual_value, '예측값': pred[h-1]})
    return pd.DataFrame(records)
